In [1]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from scipy import stats

df = pd.read_csv("full_hit_data.csv")

# Handedness
df["batterHand"] = df["batterHand"].map({"R":0, "L":1})

features = [
    "exitVelocity",
    "launchAngle",
    "sprayAngle",
    "balls",
    "strikes",
    "outs",
    "batterHand"
]

target = "slg"

train = df.dropna(subset=features + [target])

model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=6,
    random_state=42
)

model.fit(train[features], train[target])

df["xSLG"] = model.predict(df[features])

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003981 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 778
[LightGBM] [Info] Number of data points in the train set: 560353, number of used features: 7
[LightGBM] [Info] Start training from score 1.318497


In [7]:
hitters = df.groupby("Batter").agg(
    PA=("Batter", "count"),
    Team=("hittingTeam", "first"),
    xSLG=("xSLG", "mean"),
    SLG=("slg", "mean"),
    EV=("exitVelocity", "mean"),
    LA=("launchAngle", "mean")
).reset_index()

# only relevant hitters
hitters = hitters[hitters["PA"] >= 25]

In [8]:
hitters["xSLG+"] = (
    100 +
    10 * stats.zscore(hitters["xSLG"])
)

hitters["xSLG+"] = (
    hitters["xSLG+"]
    .round()
    .astype(int)
)

In [12]:
leaderboard = hitters.sort_values(
    "xSLG+",
    ascending=False
)

In [13]:
leaderboard[leaderboard["Team"] == "URI"].sort_values(by="xSLG+", ascending=False)

,Batter,PA,Team,xSLG,SLG,EV,LA,xSLG+
195,A. DePino,131,URI,1.775915,1.786260,95.795037,12.743821,119
9639,R. Butler,55,URI,1.533373,1.400000,90.114741,8.996618,109
2301,C. Grotyohann,31,URI,1.502255,1.354839,91.865953,9.612757,108
3700,D. Perron,104,URI,1.477768,1.461538,90.783531,7.673634,107
7829,M. Anderson,343,URI,1.464695,1.478134,95.288687,7.922553,106
5828,J. Hopko,164,URI,1.456794,1.500000,92.161698,6.606951,106
9785,R. Henschel,54,URI,1.441651,1.333333,88.039527,9.438035,105
996,B. Butler,58,URI,1.422795,1.224138,95.819672,7.399918,105
527,A. Medina,25,URI,1.345607,1.320000,83.767298,1.318444,102
4055,E. Genther,151,URI,1.260657,1.238411,93.035685,2.028131,98


In [14]:
leaderboard.head()

,Batter,PA,Team,xSLG,SLG,EV,LA,xSLG+
3651,D. Needham,35,WCU,2.417936,2.342857,100.445124,20.338241,145
5928,J. Kinker,27,FGCU,2.341609,2.407407,98.242672,17.930497,142
8525,M. Sears,41,COFC,2.335865,2.292683,95.259647,17.897638,141
2980,C. Stark,43,TENN,2.268930,2.162791,95.142807,18.773422,139
10341,S. Lewis,103,TROY,2.260962,2.291262,99.887179,16.826170,138
